In [1]:
import os
import numpy as np
import pandas as pd

from scipy import stats

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("darkgrid")



In [2]:
BASE_DIR_CANDIDATES = [
    "/kaggle/input/tabular-playground-series-dec-2021",
    "/kaggle/data/tabular-playground-series-dec-2021",
    "/kaggle/input",
    "/kaggle/data",
]
DATA_DIR = None
for d in BASE_DIR_CANDIDATES:
    if os.path.exists(os.path.join(d, "train.csv")) and os.path.exists(
        os.path.join(d, "test.csv")
    ):
        DATA_DIR = d
        break

if DATA_DIR is None:
    raise FileNotFoundError(
        "Could not find train.csv/test.csv under expected Kaggle paths. "
        f"Tried: {BASE_DIR_CANDIDATES}"
    )

train_path = os.path.join(DATA_DIR, "train.csv")
test_path = os.path.join(DATA_DIR, "test.csv")
sample_sub_path = os.path.join(DATA_DIR, "sample_submission.csv")

train_path, test_path, sample_sub_path



('/kaggle/input/tabular-playground-series-dec-2021/train.csv',
 '/kaggle/input/tabular-playground-series-dec-2021/test.csv',
 '/kaggle/input/tabular-playground-series-dec-2021/sample_submission.csv')

In [3]:
n_threads = str(os.cpu_count() or 1)
os.environ.setdefault("OMP_NUM_THREADS", n_threads)
os.environ.setdefault("MKL_NUM_THREADS", n_threads)
os.environ.setdefault("OPENBLAS_NUM_THREADS", n_threads)
os.environ.setdefault("VECLIB_MAXIMUM_THREADS", n_threads)
os.environ.setdefault("NUMEXPR_NUM_THREADS", n_threads)

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

target_col = "Cover_Type"
id_col = "Id"

# Bugfix: pandas 'pyarrow' engine does not support nrows=0 (used for header read).
# Read headers with default engine, then use pyarrow (if available) for the full reads.
read_csv_kwargs = {}
try:
    import pyarrow  # noqa: F401

    read_csv_kwargs["engine"] = "pyarrow"
except Exception:
    read_csv_kwargs["low_memory"] = False

train_head = pd.read_csv(train_path, nrows=0)  # default engine for header
test_head = pd.read_csv(test_path, nrows=0)  # default engine for header
train_cols = train_head.columns.tolist()
test_cols = test_head.columns.tolist()

feature_cols = [c for c in test_cols if c != target_col]  # test has no target
if id_col not in feature_cols:
    raise ValueError(f"{id_col} not found in test columns.")
usecols_train = feature_cols + [target_col]
usecols_test = feature_cols

train = pd.read_csv(train_path, usecols=usecols_train, **read_csv_kwargs)
test = pd.read_csv(test_path, usecols=usecols_test, **read_csv_kwargs)
submission = pd.read_csv(sample_sub_path, usecols=[id_col], **read_csv_kwargs)

X = train.drop(columns=[target_col])
y_raw = train[target_col]
X_test = test  # no copy; we never mutate X_test

feature_cols = [c for c in X.columns if c != id_col]

y_raw = pd.to_numeric(y_raw, errors="coerce")
if y_raw.isna().any():
    raise ValueError("Found NaN in target after coercion; cannot train.")
y_raw = y_raw.astype(int)

classes_sorted = np.sort(y_raw.unique())
class_to_idx = {c: i for i, c in enumerate(classes_sorted)}
idx_to_class = {i: c for c, i in class_to_idx.items()}
y = y_raw.map(class_to_idx).astype(np.int32)

X_feat = X[feature_cols].to_numpy(dtype=np.float32, copy=False)
X_test_feat = X_test[feature_cols].to_numpy(dtype=np.float32, copy=False)
y_arr = y.to_numpy(dtype=np.int32, copy=False)

test_ids = X_test[id_col].to_numpy(copy=False)

SEEDS = [0, 1, 2, 3, 4]  # 5 base predictors to vote

pd.Series(y_arr).value_counts().head(), len(classes_sorted)



(1    2036254
 0    1320866
 2     176184
 6      56125
 5      10237
 Name: count, dtype: int64,
 7)

In [4]:
train_mean = X_feat.mean(axis=0, dtype=np.float64)
train_var = X_feat.var(axis=0, dtype=np.float64)  # ddof=0
train_scale = np.sqrt(train_var, dtype=np.float64)
train_scale[train_scale == 0.0] = 1.0

train_mean32 = train_mean.astype(np.float32, copy=False)
train_scale32 = train_scale.astype(np.float32, copy=False)

X_feat_std = np.array(X_feat, dtype=np.float32, copy=True)
X_feat_std -= train_mean32
X_feat_std /= train_scale32

X_test_feat_std = np.array(X_test_feat, dtype=np.float32, copy=True)
X_test_feat_std -= train_mean32
X_test_feat_std /= train_scale32

n = X_feat_std.shape[0]
all_idx = np.arange(n, dtype=np.int32)

try:
    base_tr_idx, base_val_idx = train_test_split(
        all_idx,
        test_size=0.10,
        random_state=0,
        stratify=y_arr,
    )
except ValueError:
    base_tr_idx, base_val_idx = train_test_split(
        all_idx,
        test_size=0.10,
        random_state=0,
        stratify=None,
    )

SUBSET_FRAC = 0.35  # chosen to fit within 600s on CPU; deterministic and applied equally to all seeds
subset_size = int(len(base_tr_idx) * SUBSET_FRAC)
if subset_size < 1:
    subset_size = len(base_tr_idx)

splits = {}
for seed in SEEDS:
    rng = np.random.RandomState(seed)
    tr_idx = base_tr_idx.copy()
    rng.shuffle(tr_idx)
    tr_idx = tr_idx[:subset_size]
    splits[seed] = (tr_idx, base_val_idx)

n_test = X_test_feat_std.shape[0]
n_models = len(SEEDS)
pred_mat = np.empty((n_test, n_models), dtype=np.int32)

for i, seed in enumerate(SEEDS):
    tr_idx, _ = splits[seed]
    X_tr = X_feat_std[tr_idx]
    y_tr = y_arr[tr_idx]

    clf = LogisticRegression(
        multi_class="multinomial",
        solver="lbfgs",
        max_iter=200,
        n_jobs=-1,
        random_state=seed,
    )
    clf.fit(X_tr, y_tr)
    pred_mat[:, i] = clf.predict(X_test_feat_std).astype(np.int32, copy=False)

results = pd.DataFrame({id_col: test_ids})
for i in range(n_models):
    results[f"p{i+1}"] = pred_mat[:, i]

print(results.shape)
results.head()



/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _c

(400000, 6)


,Id,p1,p2,p3,p4,p5
0,814683,1,1,1,1,1
1,1357371,1,1,1,1,1
2,2106112,0,0,0,0,0
3,3483684,0,0,0,0,0
4,3960754,6,6,6,6,6


In [5]:
vote_cols = [c for c in results.columns if c.startswith("p")]
pred_mat = results[vote_cols].to_numpy(dtype=np.int32, copy=False)


def majority_vote_rows(a: np.ndarray, n_classes: int) -> np.ndarray:
    n_rows, n_votes = a.shape
    counts = np.zeros((n_rows, n_classes), dtype=np.int16)
    rows = np.arange(n_rows)
    for j in range(n_votes):  # tiny fixed loop (5)
        counts[rows, a[:, j]] += 1
    return counts.argmax(axis=1).astype(np.int32, copy=False)


results["ensemble_idx"] = majority_vote_rows(pred_mat, n_classes=len(classes_sorted))
results["ensemble"] = results["ensemble_idx"].map(idx_to_class).astype(int)
results.head()




,Id,p1,p2,p3,p4,p5,ensemble_idx,ensemble
0,814683,1,1,1,1,1,1,2
1,1357371,1,1,1,1,1,1,2
2,2106112,0,0,0,0,0,0,1
3,3483684,0,0,0,0,0,0,1
4,3960754,6,6,6,6,6,6,7


In [6]:
def nunique(a, axis):
    return (np.diff(np.sort(a, axis=axis), axis=axis) != 0).sum(axis=axis) + 1




In [7]:
results["dif"] = nunique(pred_mat, 1) - 1
results.head()



,Id,p1,p2,p3,p4,p5,ensemble_idx,ensemble,dif
0,814683,1,1,1,1,1,1,2,0
1,1357371,1,1,1,1,1,1,2,0
2,2106112,0,0,0,0,0,0,1,0
3,3483684,0,0,0,0,0,0,1,0
4,3960754,6,6,6,6,6,6,7,0


In [8]:
results.dif.value_counts()



dif
0    398295
1      1699
2         6
Name: count, dtype: int64

In [9]:
# Correctness: submission must be Id,Cover_Type with the test Ids in the same order as sample_submission.
submission[target_col] = results["ensemble"].to_numpy(dtype=np.int64, copy=False)
submission[[id_col, target_col]].to_csv("submission.csv", index=False)
submission.head()

,Id,Cover_Type
0,814683,2
1,1357371,2
2,2106112,1
3,3483684,1
4,3960754,7
